In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import timedelta
import math

plt.rcParams['font.family'] ='Malgun Gothic'
plt.rcParams['axes.unicode_minus'] =False

# 실행결과 경고메시지 출력 제외
import warnings
warnings.filterwarnings('ignore')

# 데이터 불러오기 및 병합

In [2]:
oxidation = pd.read_csv("../B1_data/raw_data/Oxidation.csv", index_col = 'No_Die')
photo_1 = pd.read_csv("../B1_data/raw_data/Photo_softbake.csv", index_col = 'No_Die')
photo_2 = pd.read_csv("../B1_data/raw_data/Photo_lithograpy.csv", index_col = 'No_Die')
etching = pd.read_csv("../B1_data/raw_data/Etching.csv", index_col = 'No_Die')
ion = pd.read_csv("../B1_data/raw_data/Ion_Implantation.csv", index_col = 'No_Die')
inspect = pd.read_csv("../B1_data/raw_data/Inspect.csv", index_col = 'No_Die')

In [3]:
# 겹치는 행 삭제 : No_Die, Lot_Num, Wafer_Num, process, Datetime

oxi_1 = oxidation.drop(columns = ['Lot_Num', 'Wafer_Num', 'process', 'Datetime'])
photo_1_1 = photo_1.drop(columns = ['Lot_Num', 'Wafer_Num', 'process 2', 'Datetime'])
photo_2_1 = photo_2.drop(columns = ['Lot_Num', 'Wafer_Num', 'Process 2-1', 'Datetime'])
etch_1 = etching.drop(columns = ['Lot_Num', 'Wafer_Num', 'Process 3', 'Datetime'])
ion_1 = ion.drop(columns = ['Lot_Num', 'Wafer_Num', 'process4', 'Datetime'])
inspect_1 = inspect.drop(columns = ['Wafer_map', 'Error_message'])

display(oxi_1.head())
display(photo_1_1.head())
display(photo_2_1.head())
display(etch_1.head())
display(ion_1.head())
display(inspect_1.head())

,Ox_Chamber,type,Temp_OXid,Vapor,ppm,Pressure,Oxid_time,thickness
No_Die,,,,,,,,
NOLSM32513132,2,dry,1214.33,O2,26.78,0.30,120,712.98
NOLSM32513133,2,dry,977.98,O2,30.90,0.14,137,714.41
NOLSM32513134,2,dry,1175.95,O2,31.11,0.25,116,710.27
NOLSM32513135,2,dry,933.47,O2,31.20,0.29,143,710.62
NOLSM32513136,2,wet,1140.60,H2O,31.38,0.20,76,711.70


,photo_soft_Chamber,resist_target,N2_HMDS,pressure_HMDS,temp_HMDS,temp_HMDS_bake,time_HMDS_bake,spin1,spin2,spin3,photoresist_bake,temp_softbake,time_softbake
No_Die,,,,,,,,,,,,,
NOLSM32513132,1,1.426,17.362,15.112,19.893,200.511,89.963,502.276,4017.015,4903.689,5.105,91.947,29.911
NOLSM32513133,1,0.730,16.015,15.083,20.035,199.876,89.988,507.132,4073.049,4979.083,4.920,91.073,30.004
NOLSM32513134,1,0.903,19.229,14.917,19.884,202.098,89.918,501.579,4085.002,5031.775,4.948,92.076,30.037
NOLSM32513135,1,0.510,18.894,14.886,20.023,194.679,90.094,503.475,4045.281,4969.189,4.808,91.422,30.037
NOLSM32513136,1,1.696,13.851,14.705,20.000,202.137,90.070,501.384,4010.123,5092.433,5.056,94.836,30.087


,lithography_Chamber,Line_CD,UV_type,Wavelength,Resolution,Energy_Exposure
No_Die,,,,,,
NOLSM32513132,1,41.548,I,365,505.405,109.653
NOLSM32513133,1,53.649,H,405,541.691,104.946
NOLSM32513134,1,47.647,I,365,532.233,106.563
NOLSM32513135,1,33.040,G,436,537.622,108.696
NOLSM32513136,1,57.411,H,405,522.135,109.944


,Etching_Chamber,Thin F4,Thin F3,Thin F2,Thin F1,Temp_Etching,Source_Power,Selectivity
No_Die,,,,,,,,
NOLSM32513132,1,340.0,1522.0,3644.0,5732.0,72.628,52.080,1.188
NOLSM32513133,2,265.0,1513.0,3631.0,5729.0,70.220,52.028,0.847
NOLSM32513134,3,411.0,1568.0,3653.0,5729.0,71.140,50.705,1.152
NOLSM32513135,1,328.0,1326.0,3661.0,5718.0,71.306,51.550,1.063
NOLSM32513136,2,219.0,1451.0,3637.0,5729.0,72.982,50.681,1.120


,Chamber_Num,Flux60s,Flux90s,Flux160s,Flux480s,Flux840s,input_Energy,Temp_implantation,Furance_Temp,RTA_Temp
No_Die,,,,,,,,,,
NOLSM32513132,1,1.500000e+16,1.320000e+17,6.470000e+17,3.010000e+17,6.000000e+17,30795.856,103.513,854.0,154
NOLSM32513133,2,1.110000e+16,4.370000e+16,1.040000e+18,3.030000e+17,6.000000e+17,32135.659,105.628,895.0,156
NOLSM32513134,3,1.040000e+16,1.510000e+16,6.470000e+17,2.980000e+17,6.000000e+17,31057.876,102.716,898.0,152
NOLSM32513135,1,8.890000e+15,1.020000e+17,3.410000e+17,3.000000e+17,6.000000e+17,32140.435,102.902,879.0,155
NOLSM32513136,2,1.670000e+16,7.880000e+16,7.260000e+17,3.020000e+17,6.000000e+17,31985.989,101.465,882.0,155


NameError: name 'inpect_1' is not defined

In [4]:
full_data = pd.concat([oxi_1,photo_1_1,photo_2_1,etch_1,ion_1,inspect_1], axis = 1)
full_data = full_data.reset_index(drop=False)
#full_data.columns.tolist()

In [5]:
full_data.to_csv("./full_data.csv", index = False)

# 데이터 이상치 및 결측치 제거

## 결측치 처리

In [6]:
full_data.isnull().sum()

No_Die                  0
Ox_Chamber              0
type                    0
Temp_OXid               0
Vapor                   0
ppm                     0
Pressure                0
Oxid_time               0
thickness               0
photo_soft_Chamber      0
resist_target           0
N2_HMDS                 0
pressure_HMDS           0
temp_HMDS               0
temp_HMDS_bake          0
time_HMDS_bake          0
spin1                   0
spin2                   0
spin3                   0
photoresist_bake        0
temp_softbake           0
time_softbake           0
lithography_Chamber     0
Line_CD                18
UV_type                 0
Wavelength              0
Resolution             18
Energy_Exposure        18
Etching_Chamber         0
Thin F4                18
Thin F3                27
Thin F2                18
Thin F1                54
Temp_Etching            0
Source_Power            0
Selectivity             0
Chamber_Num             0
Flux60s                27
Flux90s     

In [7]:
# 여러개의 변수가 Nan인 경우 삭제

line_cd_index = full_data[full_data['Line_CD'].isna()].index
full_data = full_data.drop(line_cd_index)
full_data.isnull().sum()

No_Die                  0
Ox_Chamber              0
type                    0
Temp_OXid               0
Vapor                   0
ppm                     0
Pressure                0
Oxid_time               0
thickness               0
photo_soft_Chamber      0
resist_target           0
N2_HMDS                 0
pressure_HMDS           0
temp_HMDS               0
temp_HMDS_bake          0
time_HMDS_bake          0
spin1                   0
spin2                   0
spin3                   0
photoresist_bake        0
temp_softbake           0
time_softbake           0
lithography_Chamber     0
Line_CD                 0
UV_type                 0
Wavelength              0
Resolution              0
Energy_Exposure         0
Etching_Chamber         0
Thin F4                 0
Thin F3                 9
Thin F2                 0
Thin F1                36
Temp_Etching            0
Source_Power            0
Selectivity             0
Chamber_Num             0
Flux60s                 9
Flux90s     

#### Etching - Thin F1, F4의 경우 계측누락으로 간주하여 중앙값으로 대체

In [8]:
f1_index = full_data[full_data['Thin F1'].isna()].index
f1_index

Int64Index([  462,   476,   483,   575,  2166,  2180,  2187,  2279,  3870,
             3884,  3891,  3983,  5574,  5588,  5595,  5687,  7278,  7292,
             7299,  7391,  8982,  8996,  9003,  9095, 10686, 10700, 10707,
            10799, 12390, 12404, 12411, 12503, 14094, 14108, 14115, 14207],
           dtype='int64')

In [9]:
full_data['Thin F1'] = full_data['Thin F1'].fillna(full_data['Thin F1'].median())
full_data['Thin F3'] = full_data['Thin F1'].fillna(full_data['Thin F3'].median())
#full_data.loc[f1_index, 'Thin F1']

In [10]:
full_data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 15372 entries, 0 to 15389
Data columns (total 50 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   No_Die               15372 non-null  object 
 1   Ox_Chamber           15372 non-null  int64  
 2   type                 15372 non-null  object 
 3   Temp_OXid            15372 non-null  float64
 4   Vapor                15372 non-null  object 
 5   ppm                  15372 non-null  float64
 6   Pressure             15372 non-null  float64
 7   Oxid_time            15372 non-null  int64  
 8   thickness            15372 non-null  float64
 9   photo_soft_Chamber   15372 non-null  int64  
 10  resist_target        15372 non-null  float64
 11  N2_HMDS              15372 non-null  float64
 12  pressure_HMDS        15372 non-null  float64
 13  temp_HMDS            15372 non-null  float64
 14  temp_HMDS_bake       15372 non-null  float64
 15  time_HMDS_bake       15372 non-null 

## 이상치처리

### Oxidation

In [11]:
# pressure가 음수인 경우 삭제

drop_pressure_index = full_data[full_data['Pressure'] < 0].index
oxi_preprocessing = full_data.drop(drop_pressure_index)
print(len(oxi_preprocessing))

15116


In [12]:
# oxid_time이 0 이하인 경우 삭제

drop_time_index = oxi_preprocessing[oxi_preprocessing['Oxid_time'] <= 0].index
oxi_preprocessing = oxi_preprocessing.drop(drop_time_index)
print(len(oxi_preprocessing))

15033


### Photo_1

In [13]:
# resist_target이 0 이하인 경우 삭제

drop_regtar_index = oxi_preprocessing[oxi_preprocessing['resist_target'] <= 0].index
photo1_preprocessing = oxi_preprocessing.drop(drop_regtar_index)
print(len(photo1_preprocessing))

15029


### Ion

In [14]:
# Flux90, 160s 에서 음수인 경우 삭제

drop_flux_index = photo1_preprocessing[(photo1_preprocessing['Flux90s'] <0) | (photo1_preprocessing['Flux160s'] <0)].index
ion_preprocessing = photo1_preprocessing.drop(drop_flux_index)
print(len(ion_preprocessing))

14816


In [15]:
preprocessing_done = ion_preprocessing.copy()
preprocessing_done = preprocessing_done.reset_index(drop=True)

### 파생변수 생성

#### chamber path

In [16]:
# Etching과 ion 공정의 path가 같으므로 한개 제외

preprocessing_done['c_path'] = preprocessing_done[['Ox_Chamber','photo_soft_Chamber',
                                 'lithography_Chamber','Etching_Chamber']].astype(str).agg(''.join, axis = 1)
preprocessing_done[['Ox_Chamber','photo_soft_Chamber',
                                 'lithography_Chamber','Etching_Chamber','c_path']].head()

,Ox_Chamber,photo_soft_Chamber,lithography_Chamber,Etching_Chamber,c_path
0,2,1,1,1,2111
1,2,1,1,2,2112
2,2,1,1,3,2113
3,2,1,1,1,2111
4,2,1,1,2,2112


#### kper_na
- 해상도(Resolution) 공정상수 (K x λ)/NA (λ = wavelength - 빛의 파장 \ (NA = 렌즈의 개구수)
- 해상도와 wavelength의 관계식을 이용하여 만든 파생변수
- Energy Exposure과 Resolution의 다중공선성 문제를 해결하기 위한 방법

In [17]:
preprocessing_done['kperna'] = preprocessing_done['Resolution'] / preprocessing_done['Wavelength']
preprocessing_done[['kperna','Resolution', 'Wavelength']].head()

,kperna,Resolution,Wavelength
0,1.384671,505.405,365
1,1.337509,541.691,405
2,1.458173,532.233,365
3,1.233078,537.622,436
4,1.289222,522.135,405


#### Thin's 를 사용한 시간에 따른 속도 = etching_velo
- (Thin F1 - Thin F4) / (30 * 60)
- 실제 공정에서 컨트롤할 수 있는 요인으로 엣칭 속도를 선정하여 사용하고 있기때문에 파생변수로 생성함

In [18]:
preprocessing_done['etching_velo'] = (preprocessing_done['Thin F1'] - preprocessing_done['Thin F4']) / 1800                         
preprocessing_done[['etching_velo','Thin F1', 'Thin F4']].head()

,etching_velo,Thin F1,Thin F4
0,2.995556,5732.0,340.0
1,3.035556,5729.0,265.0
2,2.954444,5729.0,411.0
3,2.994444,5718.0,328.0
4,3.061111,5729.0,219.0


#### 이온주입공정시 목표 이온 주입량 = max_flux 
- max_flux = max(Flux 160s, Flux 480s)

In [19]:
preprocessing_done['max_flux'] = preprocessing_done[['Flux160s', 'Flux480s']].max(axis=1)
preprocessing_done[['max_flux','Flux160s','Flux480s']].head()

,max_flux,Flux160s,Flux480s
0,6.470000e+17,6.470000e+17,3.010000e+17
1,1.040000e+18,1.040000e+18,3.030000e+17
2,6.470000e+17,6.470000e+17,2.980000e+17
3,3.410000e+17,3.410000e+17,3.000000e+17
4,7.260000e+17,7.260000e+17,3.020000e+17


### 번외: control chart용

In [20]:
for_c_chart = preprocessing_done[['Target','Datetime','c_path']]
for_c_chart['c_path'] = for_c_chart['c_path'].astype(str)
for_c_chart.to_csv("./for_c_chart.csv", index = False)
for_c_chart.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14816 entries, 0 to 14815
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Target    14816 non-null  int64 
 1   Datetime  14816 non-null  object
 2   c_path    14816 non-null  object
dtypes: int64(1), object(2)
memory usage: 347.4+ KB


## 필요없는 열 제거

In [21]:
preprocessing_done.columns

Index(['No_Die', 'Ox_Chamber', 'type', 'Temp_OXid', 'Vapor', 'ppm', 'Pressure',
       'Oxid_time', 'thickness', 'photo_soft_Chamber', 'resist_target',
       'N2_HMDS', 'pressure_HMDS', 'temp_HMDS', 'temp_HMDS_bake',
       'time_HMDS_bake', 'spin1', 'spin2', 'spin3', 'photoresist_bake',
       'temp_softbake', 'time_softbake', 'lithography_Chamber', 'Line_CD',
       'UV_type', 'Wavelength', 'Resolution', 'Energy_Exposure',
       'Etching_Chamber', 'Thin F4', 'Thin F3', 'Thin F2', 'Thin F1',
       'Temp_Etching', 'Source_Power', 'Selectivity', 'Chamber_Num', 'Flux60s',
       'Flux90s', 'Flux160s', 'Flux480s', 'Flux840s', 'input_Energy',
       'Temp_implantation', 'Furance_Temp', 'RTA_Temp', 'Lot_Num', 'Wafer_Num',
       'Datetime', 'Target', 'c_path', 'kperna', 'etching_velo', 'max_flux'],
      dtype='object')

In [22]:
# 라벨생성

def pn_label(value):
    if value < 195:
        return 0
    else:
        return 1

preprocessing_done['y_label'] = preprocessing_done['Target'].apply(pn_label)
preprocessing_done[['Target', 'y_label']].head(20)

,Target,y_label
0,141,0
1,55,0
2,96,0
3,105,0
4,79,0
5,96,0
6,115,0
7,167,0
8,84,0
9,30,0


In [24]:
preprocessing_done.columns

Index(['No_Die', 'Ox_Chamber', 'type', 'Temp_OXid', 'Vapor', 'ppm', 'Pressure',
       'Oxid_time', 'thickness', 'photo_soft_Chamber', 'resist_target',
       'N2_HMDS', 'pressure_HMDS', 'temp_HMDS', 'temp_HMDS_bake',
       'time_HMDS_bake', 'spin1', 'spin2', 'spin3', 'photoresist_bake',
       'temp_softbake', 'time_softbake', 'lithography_Chamber', 'Line_CD',
       'UV_type', 'Wavelength', 'Resolution', 'Energy_Exposure',
       'Etching_Chamber', 'Thin F4', 'Thin F3', 'Thin F2', 'Thin F1',
       'Temp_Etching', 'Source_Power', 'Selectivity', 'Chamber_Num', 'Flux60s',
       'Flux90s', 'Flux160s', 'Flux480s', 'Flux840s', 'input_Energy',
       'Temp_implantation', 'Furance_Temp', 'RTA_Temp', 'Lot_Num', 'Wafer_Num',
       'Datetime', 'Target', 'c_path', 'kperna', 'etching_velo', 'max_flux',
       'y_label'],
      dtype='object')

In [27]:
# 모델 학습 데이터 생성
final_data = preprocessing_done[['type', 'Temp_OXid', 'ppm', 'Pressure', 'Oxid_time', 
                                'N2_HMDS', 'pressure_HMDS', 'temp_HMDS', 'time_HMDS_bake', 
                                'spin1', 'spin2', 'spin3', 'photoresist_bake', 'temp_softbake', 
                                'time_softbake', 'kperna', 'Energy_Exposure', 'Source_Power', 'Temp_Etching', 
                                'etching_velo', 'max_flux', 'input_Energy', 'Temp_implantation', 'Furance_Temp', 
                                'RTA_Temp', 'y_label']]

final_data.to_csv("../B1_data/preprocessing_data/final_data.csv", index = False)

In [29]:
# 1차 필요없는 열 제거

drop_cols_df = preprocessing_done.drop(columns = ['Vapor','Ox_Chamber','photo_soft_Chamber','lithography_Chamber',
                                                  'Etching_Chamber', 'Wavelength', 'Resolution', 'Chamber_Num', 
                                                  'Thin F4', 'Thin F3', 'Thin F2', 'Thin F1', 
                                                  'Lot_Num', 'Wafer_Num', 'No_Die', 'Flux60s','Flux90s',
                                                  'Flux160s', 'Flux480s', 'Flux840s', 'Datetime', 'Target', 'c_path'])
drop_cols_df.to_csv("../B1_data/preprocessing_data/test.csv", index = False)

In [26]:
# 필요없는 열 제거 + 타겟 제외

target_cols_df = preprocessing_done.drop(columns = ['Vapor','Ox_Chamber','photo_soft_Chamber','lithography_Chamber',
                                                  'Etching_Chamber', 'Wavelength', 'Resolution', 'Chamber_Num', 
                                                  'Thin F4', 'Thin F3', 'Thin F2', 'Thin F1', 
                                                  'Lot_Num', 'Wafer_Num', 'No_Die', 'Flux60s','Flux90s',
                                                  'Flux160s', 'Flux480s', 'Flux840s', 'Datetime'])

target_cols_df.to_csv("../B1_data/preprocessing_data/preprocessing_include_target.csv", index = False)

In [27]:
# 선정된 열만 있는 데이터 
done_df = drop_cols_df[['Temp_OXid','ppm','type','UV_type','thickness','resist_target','temp_HMDS_bake','kperna',
                        'Line_CD','Energy_Exposure','Source_Power','Temp_Etching',
                        'etching_velo','RTA_Temp','Temp_implantation','input_Energy',
                        'max_flux','y_label']]
                        
done_df.to_csv("../B1_data/preprocessing_data/preprocessing_data.csv", index=False)

In [28]:
# 월/일 분석용 데이터

for_analysis_date = preprocessing_done[['Target','Datetime']]
for_analysis_date.to_csv("../B1_data/preprocessing_data/for_analysis_date.csv", index = False)